# Schwinger hadron challenge — Part 1: physics you can verify

Ex 1.1–1.5. Imports the Part 0 answers from `schwinger_core.py`. Each exercise ends with the values to paste into the grader.

Runtime: ~2 min total (Ex 1.5 needs `qiskit-aer`, ~35 s).

In [1]:
import math, time, warnings
import numpy as np
from scipy.sparse.linalg import eigsh, expm_multiply
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector

from challenge_utils import trotter_step_electric_2q
from schwinger_core import *
REF_DIR = 'reference_data'

## Ex 1.1 — Hamiltonian builders

In [2]:
def pauli_op(n: int, ops: dict[int, str], coeff: float = 1.0) -> SparsePauliOp:
    """SparsePauliOp on n qubits with the Paulis in ops = {site: 'X'|'Y'|'Z'} (little-endian handled here)."""
    s = ["I"] * n
    for j, p in ops.items():
        s[n - 1 - j] = p
    return SparsePauliOp.from_list([("".join(s), coeff)])


def mass_hamiltonian(L: int, m: float = 0.5) -> SparsePauliOp:
    """H_m = m/2 sum_j [(-1)^j Z_j + I]  (identity term included)."""
    n = 2 * L
    H = 0 * pauli_op(n, {})
    for j in range(n):
        H += pauli_op(n, {j: "Z"}, m / 2 * (-1) ** j) + pauli_op(n, {}, m / 2)
    return H.simplify()


def kinetic_hamiltonian(L: int) -> SparsePauliOp:
    """H_kin = 1/4 sum_j (X_j X_{j+1} + Y_j Y_{j+1})."""
    n = 2 * L
    H = 0 * pauli_op(n, {})
    for j in range(n - 1):
        H += pauli_op(n, {j: "X", j + 1: "X"}, 0.25) + pauli_op(n, {j: "Y", j + 1: "Y"}, 0.25)
    return H.simplify()


def electric_hamiltonian_full(L: int, g: float = 0.3) -> SparsePauliOp:
    """H_el = g^2/2 sum_{j=0}^{2L-2} (sum_{k<=j} Q_k)^2 with Q_k = -1/2 (Z_k + (-1)^k I)."""
    n = 2 * L
    H = 0 * pauli_op(n, {})
    for j in range(n - 1):
        S = 0 * pauli_op(n, {})
        for k in range(j + 1):
            S += pauli_op(n, {k: "Z"}, -0.5) + pauli_op(n, {}, -0.5 * (-1) ** k)
        H += (g**2 / 2) * (S @ S)
    return H.simplify()


def electric_hamiltonian_truncated(L: int, g: float = 0.3) -> SparsePauliOp:
    """H_el^{(Q=0)}(1): range-1 truncated, charge-zero-sector electric Hamiltonian (even L, identity dropped)."""
    assert L % 2 == 0, "the truncated electric Hamiltonian formula assumes even L"
    n = 2 * L
    half = L // 2
    H = 0 * pauli_op(n, {})
    c = g**2 / 2
    Z = lambda j: pauli_op(n, {j: "Z"})
    ZZ = lambda i, j: pauli_op(n, {i: "Z", j: "Z"})
    for k in range(half):
        H += c * (half - 0.75 - k) * ZZ(2 * k, 2 * k + 1)
        H += c * (k + 0.25) * ZZ(L + 2 * k, L + 2 * k + 1)
    for k in range(1, half - 1):
        H += c * 0.5 * (2 * Z(2 * k) + Z(2 * k + 1) - Z(L + 2 * k) - 2 * Z(L + 2 * k + 1))
    H += c * 0.5 * (2 * Z(0) + Z(1) + Z(L - 2) - Z(L + 1) - Z(2 * L - 2) - 2 * Z(2 * L - 1))
    for k in range(half - 1):
        H += c * (half - 1.25 - k) * (ZZ(2 * k, 2 * k + 2) + ZZ(2 * k + 1, 2 * k + 2))
        H += c * (half - 1.75 - k) * (ZZ(2 * k, 2 * k + 3) + ZZ(2 * k + 1, 2 * k + 3))
        H += c * (k + 0.25) * (ZZ(L + 2 * k + 2, L + 2 * k) + ZZ(L + 2 * k + 3, L + 2 * k))
        H += c * (k + 0.75) * (ZZ(L + 2 * k + 2, L + 2 * k + 1) + ZZ(L + 2 * k + 3, L + 2 * k + 1))
    return H.simplify()


def schwinger_hamiltonian(L: int, m: float = 0.5, g: float = 0.3, truncated: bool = False) -> SparsePauliOp:
    """H = H_m + H_kin + H_el  (truncated=True uses H_el^{(Q=0)}(1))."""
    H = 0 * pauli_op(2 * L, {})
    H += mass_hamiltonian(L, m)
    H += kinetic_hamiltonian(L)
    H += electric_hamiltonian_truncated(L, g) if truncated else electric_hamiltonian_full(L, g)
    return H.simplify()

In [3]:
def total_charge(L: int) -> SparsePauliOp:
    n = 2 * L
    return sum((pauli_op(n, {k: "Z"}, -0.5) + pauli_op(n, {}, -0.5 * (-1) ** k)) for k in range(n)).simplify()

E0, ground_states = {}, {}
for LL in (4, 6, 8):
    Hf = schwinger_hamiltonian(LL)
    Q = total_charge(LL)
    comm = (Hf @ Q - Q @ Hf).simplify()
    w, v = eigsh(Hf.to_matrix(sparse=True), k=1, which='SA', tol=1e-12)
    E0[LL], ground_states[LL] = float(w[0]), v[:, 0]
    gs = Statevector(v[:, 0])
    print(f'L={LL}: ||[H,Q]|| = {np.linalg.norm(comm.coeffs):.1e}, <Q^2> = {np.real(gs.expectation_value(Q @ Q)):.1e}, E0 = {E0[LL]:.6f}')

# truncated == full electric term inside the Q=0 sector (up to a constant) at L=4, where nothing is truncated yet
idx = np.arange(2**8); Qd = sum(-0.5 * ((1 - 2 * ((idx >> k) & 1)) + (-1) ** k) for k in range(8)); sel = np.abs(Qd) < 1e-9
diff = (electric_hamiltonian_full(4).to_matrix(sparse=True).diagonal() - electric_hamiltonian_truncated(4).to_matrix(sparse=True).diagonal())[sel]
print('L=4 truncated vs full (Q=0) spread:', np.ptp(diff.real))

E0_L8 = E0[8]
print('\n--- paste into the grader ---')
print(f'E0_L8 = {E0_L8!r}')

L=4: ||[H,Q]|| = 0.0e+00, <Q^2> = 2.2e-16, E0 = -1.185874
L=6: ||[H,Q]|| = 0.0e+00, <Q^2> = 2.5e-16, E0 = -1.847436


L=8: ||[H,Q]|| = 0.0e+00, <Q^2> = -1.8e-14, E0 = -2.509005
L=4 truncated vs full (Q=0) spread: 8.604228440844963e-16

--- paste into the grader ---
E0_L8 = -2.509005331524565


## Ex 1.2 — electric layer is exact, the truncation is not

In [4]:
def electric_layer(L: int, t: float, g: float = 0.3) -> QuantumCircuit:
    """Circuit implementing exp(-i t H_el^{(Q=0)}(1)) (single-qubit Rz layer + barbell blocks)."""
    qc = QuantumCircuit(2 * L)
    for k in range(L // 2 - 1):
        qc.rz(g**2 * t, 2 * k)
        qc.rz(0.5 * g**2 * t, 2 * k + 1)
    qc.rz(0.5 * g**2 * t, L - 2)
    qc.rz(-0.5 * g**2 * t, L + 1)
    for k in range(1, L // 2):
        qc.rz(-0.5 * g**2 * t, L + 2 * k)
        qc.rz(-g**2 * t, L + 2 * k + 1)
    qc = trotter_step_electric_2q(qc, L, t, g)
    return qc


rng = np.random.default_rng(1)
for LL in (4, 6):
    n = 2 * LL
    layer = electric_layer(LL, 0.37, g)
    Hel = electric_hamiltonian_truncated(LL, g).to_matrix(sparse=True)
    worst = 0.0
    for _ in range(3):
        psi = rng.normal(size=2**n) + 1j * rng.normal(size=2**n); psi /= np.linalg.norm(psi)
        worst = max(worst, 1 - abs(np.vdot(Statevector(psi).evolve(layer).data, expm_multiply(-1j * 0.37 * Hel, psi))) ** 2)
    print(f'L={LL}: max infidelity electric_layer vs exp(-i t H_el^(1)) = {worst:.1e}')

L=4: max infidelity electric_layer vs exp(-i t H_el^(1)) = 1.3e-15
L=6: max infidelity electric_layer vs exp(-i t H_el^(1)) = 2.0e-15


In [5]:
L8 = 8
psi_wave_8 = Statevector(prep_wave(L8, vacuum_prep_theta_OV_1, vacuum_prep_theta_OV_3, wave_prep_theta_O_11, wave_prep_theta_O_22)).data
psi_vac_8 = Statevector(prep_vacuum(L8, vacuum_prep_theta_OV_1, vacuum_prep_theta_OV_3)).data
H8_full = schwinger_hamiltonian(L8).to_matrix(sparse=True)
H8_trunc = schwinger_hamiltonian(L8, truncated=True).to_matrix(sparse=True)

def exact_X(H_sparse, t: float, L: int = L8) -> np.ndarray:
    """Vacuum-subtracted condensate after exact evolution of prep_wave / prep_vacuum for time t."""
    return chi_from_state(expm_multiply(-1j * t * H_sparse, psi_wave_8), L) - chi_from_state(expm_multiply(-1j * t * H_sparse, psi_vac_8), L)

X_full_t4 = exact_X(H8_full, 4.0)
X_trunc_t4 = exact_X(H8_trunc, 4.0)
truncation_shift = float(np.max(np.abs(X_full_t4 - X_trunc_t4)))
print('--- paste into the grader ---')
print(f'truncation_shift = {truncation_shift!r}')

--- paste into the grader ---
truncation_shift = 0.010098067369579744


## Ex 1.3 — quality of the 2-step SC-ADAPT-VQE vacuum

In [6]:
vqe_fidelity, vqe_energy_gap, vqe_chi_error = {}, {}, {}
for LL in (4, 6, 8):
    psi_adapt = Statevector(prep_vacuum(LL, vacuum_prep_theta_OV_1, vacuum_prep_theta_OV_3))
    H_full = schwinger_hamiltonian(LL)
    vqe_fidelity[LL] = float(abs(np.vdot(ground_states[LL], psi_adapt.data)) ** 2)
    vqe_energy_gap[LL] = float(np.real(psi_adapt.expectation_value(H_full)) - E0[LL])
    vqe_chi_error[LL] = float(np.max(np.abs(chi_from_state(psi_adapt, LL) - chi_from_state(ground_states[LL], LL))))
print('--- paste into the grader ---')
print(f'vqe_fidelity = {vqe_fidelity!r}')
print(f'vqe_energy_gap = {vqe_energy_gap!r}')

--- paste into the grader ---
vqe_fidelity = {4: 0.9961025591037, 6: 0.9945245970510397, 8: 0.9929328629704}
vqe_energy_gap = {4: 0.006348285077689031, 6: 0.009390346425843354, 8: 0.012439829310504358}


## Ex 1.4 — Trotter error table, Richardson extrapolation, dominant error

In [7]:
def trotter_X(t: float, dt: float, L: int = L8) -> np.ndarray:
    n_steps = int(round(t / dt))
    qw = prep_wave(L, vacuum_prep_theta_OV_1, vacuum_prep_theta_OV_3, wave_prep_theta_O_11, wave_prep_theta_O_22)
    qv = prep_vacuum(L, vacuum_prep_theta_OV_1, vacuum_prep_theta_OV_3)
    for _ in range(n_steps):
        qw = trotter_step(qw, L, dt, m, g)
        qv = trotter_step(qv, L, dt, m, g)
    return chi_from_state(Statevector(qw), L) - chi_from_state(Statevector(qv), L)

trotter_table, X_trotter = {}, {}
X_exact_trunc = {2: exact_X(H8_trunc, 2.0), 4: exact_X(H8_trunc, 4.0)}
for tt in (2, 4):
    for dt in (1.0, 0.5, 0.25):
        X_trotter[(tt, dt)] = trotter_X(tt, dt)
        trotter_table[(tt, dt)] = float(np.max(np.abs(X_trotter[(tt, dt)] - X_exact_trunc[tt])))
X_R = (4 * X_trotter[(4, 0.25)] - X_trotter[(4, 0.5)]) / 3
richardson_error = float(np.max(np.abs(X_R - X_exact_trunc[4])))
budget = {'trotter': trotter_table[(4, 1.0)], 'truncation': truncation_shift, 'state_prep': vqe_chi_error[8]}
dominant_error = max(budget, key=budget.get)
for tt in (2, 4):
    e = [trotter_table[(tt, dt)] for dt in (1.0, 0.5, 0.25)]
    print(f't={tt}: errors {np.round(e, 4)}  ratios {e[0] / e[1]:.2f}, {e[1] / e[2]:.2f}')
print('error budget at L=8, t=4:', {k: round(v, 4) for k, v in budget.items()})
print('\n--- paste into the grader ---')
print(f'trotter_table = {trotter_table!r}')
print(f'richardson_error = {richardson_error!r}')
print(f'dominant_error = {dominant_error!r}')

t=2: errors [0.2018 0.0499 0.0124]  ratios 4.04, 4.02
t=4: errors [0.0948 0.0241 0.006 ]  ratios 3.94, 3.98
error budget at L=8, t=4: {'trotter': 0.0948, 'truncation': 0.0101, 'state_prep': 0.0383}

--- paste into the grader ---
trotter_table = {(2, 1.0): 0.20176867243681773, (2, 0.5): 0.049909192401686564, (2, 0.25): 0.012408287778455485, (4, 1.0): 0.09477142499176838, (4, 0.5): 0.024052541998538013, (4, 0.25): 0.0060370829897916645}
richardson_error = 0.00018635804434802283
dominant_error = 'trotter'


## Ex 1.5 — MPS bond-dimension convergence at L = 34, t = 8

Needs `qiskit-aer`. Wall times depend on your machine.

In [8]:
from qiskit_aer.primitives import EstimatorV2 as AerEstimatorV2

L = 34
t = 8
observables = chiral_condensate_observables(L)
qc_wave_init = prep_wave(L, vacuum_prep_theta_OV_1, vacuum_prep_theta_OV_3, wave_prep_theta_O_11, wave_prep_theta_O_22)
qc_vacuum_init = prep_vacuum_for_subtraction(L)
qc_wave, _ = evolve_circuits(qc_wave_init, L, t, m, g)
qc_vacuum, _ = evolve_circuits(qc_vacuum_init, L, t, m, g)
qc_wave_d, qc_vacuum_d = qc_wave.decompose(reps=3), qc_vacuum.decompose(reps=3)

_ref = np.load(f'{REF_DIR}/mps_reference_L34.npz')
X_64 = _ref['chi_wave_t8_bd64'] - _ref['chi_vacuum_t8_bd64']

bonds = [8, 12, 20, 40]
X_by_bond, mps_seconds = {}, {}
for chi in bonds:
    est = AerEstimatorV2(options={"backend_options": {"method": "matrix_product_state",
                                                      "matrix_product_state_max_bond_dimension": chi,
                                                      "matrix_product_state_truncation_threshold": 1e-10},
                                  "run_options": {"seed_simulator": 7}})
    t0_ = time.time()
    res = est.run([(qc_wave_d, observables), (qc_vacuum_d, observables)]).result()
    mps_seconds[chi] = round(time.time() - t0_, 1)
    X_by_bond[chi] = np.asarray(res[0].data.evs, float) - np.asarray(res[1].data.evs, float)
    print(f'bond {chi:3d}: {mps_seconds[chi]:6.1f} s   max |X - X_64| = {np.max(np.abs(X_by_bond[chi] - X_64)):.5f}')

mps_err = {chi: float(np.max(np.abs(X_by_bond[chi] - X_64))) for chi in bonds}
X_bond40 = X_by_bond[40]
assert X_bond40.shape == (68,)
print('\n--- paste into the grader ---')
print(f'mps_seconds = {mps_seconds!r}')
print(f'mps_err = {mps_err!r}')
print('X_bond40 = np.array(' + repr([float(x) for x in X_bond40]) + ')')

bond   8:    1.0 s   max |X - X_64| = 0.04862


bond  12:    1.6 s   max |X - X_64| = 0.00970


bond  20:    4.4 s   max |X - X_64| = 0.00267


bond  40:   23.0 s   max |X - X_64| = 0.00077

--- paste into the grader ---
mps_seconds = {8: 1.0, 12: 1.6, 20: 4.4, 40: 23.0}
mps_err = {8: 0.04862326420240126, 12: 0.009697929553317497, 20: 0.0026749918427898534, 40: 0.0007736486450129476}
X_bond40 = np.array([2.069336224774787e-05, 0.00011489088776350798, 8.1264917312307e-05, 0.0001226982291457901, 0.00012902437961925806, 7.66268602115927e-05, 8.796404159638893e-05, 7.92830266436173e-05, 7.159433477943455e-05, 8.918719668304931e-05, 8.812678574054988e-05, 8.061808667514381e-05, 8.97274919103852e-05, 7.138677551821182e-05, 5.184999836371773e-05, 0.0002675988968787735, 0.000241826749394769, 0.0002950973412388924, 0.0003208451601146578, 0.0005103190454612827, 0.0002446519751806431, 0.002282276928351923, 0.002095445005962926, 0.0051328913505838125, 0.0032748762014318533, 0.019613779339167303, 0.01366816213768951, 0.05796739121957439, 0.03735800454466709, 0.25048462722957643, 0.2558732203463636, 0.5100506928413786, 0.507926473465029, -0